In [3]:
import pandas as pd
import numpy as np
import datetime as dt
import sqlite3
import os

# 1. โหลดข้อมูลดิบ (ใช้ ../ ถอยออกจาก notebooks)
df = pd.read_csv('../data/raw/OnlineRetail.csv', encoding='ISO-8859-1')

# 2. Clean Data
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# 3. บันทึกลง SQLite
conn = sqlite3.connect('../data/ecommerce.db')
df_clean.to_sql('transactions', conn, if_exists='replace', index=False)
conn.close()

print("✅ Clean Data และสร้างฐานข้อมูล ecommerce.db สำเร็จ!")

✅ Clean Data และสร้างฐานข้อมูล ecommerce.db สำเร็จ!


In [4]:
import joblib
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# 1. ดึงข้อมูลจาก SQL (ใช้ ../)
conn = sqlite3.connect('../data/ecommerce.db')
df = pd.read_sql_query("SELECT * FROM transactions", conn)
conn.close()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 2. กำหนด Cutoff Date
max_date = df['InvoiceDate'].max()
cutoff_date = max_date - dt.timedelta(days=90)

df_obs = df[df['InvoiceDate'] <= cutoff_date].copy()
df_target = df[df['InvoiceDate'] > cutoff_date].copy()

# 3. สร้าง Label Churn
obs_customers = pd.DataFrame({'CustomerID': df_obs['CustomerID'].unique()})
target_customers = df_target['CustomerID'].unique()
obs_customers['Churn'] = obs_customers['CustomerID'].apply(lambda x: 1 if x not in target_customers else 0)

# 4. Feature Engineering
features = df_obs.groupby('CustomerID').agg({
    'InvoiceDate': [lambda x: (cutoff_date - x.max()).days, 
                    lambda x: (cutoff_date - x.min()).days],
    'InvoiceNo': 'nunique',
    'Quantity': ['sum', 'mean'],
    'TotalAmount': ['sum', 'mean', 'max']
}).reset_index()

features.columns = [
    'CustomerID', 'Recency', 'Tenure', 'Frequency', 
    'Total_Quantity', 'Avg_Quantity', 
    'Monetary_Total', 'Monetary_Avg', 'Monetary_Max'
]

features['Avg_Order_Value'] = features['Monetary_Total'] / features['Frequency']
features['Purchase_Frequency_Days'] = features['Tenure'] / features['Frequency']

dataset = pd.merge(features, obs_customers, on='CustomerID')

# 5. เทรนโมเดล XGBoost
X = dataset.drop(columns=['CustomerID', 'Churn'])
y = dataset['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

model = XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='logloss')
model.fit(X_train_res, y_train_res)

# 6. บันทึกไฟล์โมเดลใน src/ (ใช้ ../)
os.makedirs('../src', exist_ok=True)
joblib.dump(model, '../src/churn_xgboost_model.pkl')
joblib.dump(X.columns.tolist(), '../src/model_features.pkl')

print("✅ เทรนและบันทึกโมเดล churn_xgboost_model.pkl และ model_features.pkl สำเร็จ!")

✅ เทรนและบันทึกโมเดล churn_xgboost_model.pkl และ model_features.pkl สำเร็จ!
